# Assignment 1 – GAN Playground

**Before you start:**
1. Enable a GPU via *Runtime → Change runtime type → T4 GPU*.
2. Run the **Setup** section below.
3. Edit `.py` files in the Colab file browser (left panel).
4. Run the training cells to launch experiments.

> **Important:** Colab sessions reset on disconnect.  
> Run the *Mount Drive* cell and keep your `.py` files in Drive so edits persist.

## 0 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# TODO: Adjust this path to wherever you store the assignment folder in Drive.
import os
ASSIGNMENT_DIR = '/content/drive/MyDrive/path_to_solution'
os.makedirs(ASSIGNMENT_DIR, exist_ok=True)
%cd {ASSIGNMENT_DIR}

In [1]:
!git clone https://github.com/eranhermush/gan-assignment.git

%cd gan-assignment/skeleton

Cloning into 'gan-assignment'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 53 (delta 29), reused 36 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 44.67 KiB | 2.98 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/gan-assignment/skeleton


## 1 · Install dependencies

In [2]:
!pip install -q wandb imageio huggingface_hub
print('Dependencies installed.')

Dependencies installed.


## 2 · Log in to Weights & Biases

In [3]:
import wandb

wandb.login()
print('Logged in to W&B.')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 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


wandb: WARNING Invalid choice


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: eran-hermosh (eran-hermosh-tel-aviv-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Logged in to W&B.


## 3 · Download the datasets

In [4]:
from huggingface_hub import snapshot_download
import os

os.makedirs("data", exist_ok=True)

if not os.path.exists("data/cat"):
    print("Downloading cat dataset from Hugging Face...")
    snapshot_download(
        repo_id="saradorfman/GAN-cats",
        repo_type="dataset",
        local_dir="data",
    )
else:
    print("Cat dataset already exists, skipping.")

print("\nDone.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 282 files:   0%|          | 0/282 [00:00<?, ?it/s]


Done.


## 4 · Verify GPU

In [5]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

CUDA available: True
Device: Tesla T4


---
## Part 1 – DCGAN

Edit `models.py`, `data_loader.py`, and `vanilla_gan.py`.

In [6]:
# Quick architecture sanity check – run this after filling in models.py.
%run models.py

=== PatchDiscriminator ===
PASS: output shape torch.Size([4, 1, 4, 4])

=== CycleGenerator ===
PASS: output shape torch.Size([4, 3, 64, 64])

=== DCGenerator ===
PASS: output shape torch.Size([4, 3, 64, 64])

=== DCDiscriminator ===
PASS: output shape torch.Size([4, 1, 1, 1])


### Task 1.7 – Baseline (no DiffAugment)

In [29]:
!python vanilla_gan.py \
    --data_preprocess vanilla \
    --num_epochs 500 \
    --sample_every 200 \
    --log_step 10

Namespace(image_size=64, conv_dim=64, noise_size=100, num_epochs=500, batch_size=16, num_workers=2, lr=0.0002, beta1=0.5, beta2=0.999, data='cat/grumpifyBprocessed', data_preprocess='vanilla', use_diffaug=False, ext='*.png', checkpoint_dir='./checkpoints_vanilla', sample_dir='output/./vanilla/grumpifyBprocessed_vanilla', log_step=10, sample_every=200, checkpoint_every=400)
data/cat/grumpifyBprocessed/*.png
204
======================================== Generator ========================================
DCGenerator(
  (up_conv1): Sequential(
    (0): Upsample(scale_factor=1.0, mode='nearest')
    (1): Conv2d(100, 256, kernel_size=(4, 4), stride=(1, 1), padding=(3, 3), bias=False)
    (2): InstanceNorm2d(256, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (3): ReLU()
  )
  (up_conv2): Sequential(
    (0): Upsample(scale_factor=2.0, mode='nearest')
    (1): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (2): InstanceNorm2d(128, eps

### Task 1.7 – With DiffAugment

In [7]:
!python vanilla_gan.py \
    --data_preprocess vanilla \
    --use_diffaug \
    --num_epochs 500 \
    --sample_every 200 \
    --log_step 10

Namespace(image_size=64, conv_dim=64, noise_size=100, num_epochs=500, batch_size=16, num_workers=2, lr=0.0002, beta1=0.5, beta2=0.999, data='cat/grumpifyBprocessed', data_preprocess='vanilla', use_diffaug=True, ext='*.png', checkpoint_dir='./checkpoints_vanilla', sample_dir='output/./vanilla/grumpifyBprocessed_vanilla_diffaug', log_step=10, sample_every=200, checkpoint_every=400)
data/cat/grumpifyBprocessed/*.png
204
======================================== Generator ========================================
DCGenerator(
  (up_conv1): Sequential(
    (0): Upsample(scale_factor=1.0, mode='nearest')
    (1): Conv2d(100, 256, kernel_size=(4, 4), stride=(1, 1), padding=(3, 3), bias=False)
    (2): InstanceNorm2d(256, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (3): ReLU()
  )
  (up_conv2): Sequential(
    (0): Upsample(scale_factor=2.0, mode='nearest')
    (1): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (2): InstanceNorm2d(1

In [1]:
!git status

fatal: not a git repository (or any of the parent directories): .git


---
## Part 2 – CycleGAN

Edit `models.py` and `cycle_gan.py` before running.

### Task 2.6 – Sanity check (1,000 iters, no cycle loss)

In [8]:
!python cycle_gan.py \
    --disc patch \
    --train_iters 1000 \
    --data_preprocess vanilla

                                      Opts                                      
--------------------------------------------------------------------------------
                             image_size: 64                                     
                                   disc: patch                                  
                                    gen: cycle                                  
                             g_conv_dim: 64                                     
                             d_conv_dim: 64                                     
                                   norm: instance                               
                              init_type: naive                                  
                            train_iters: 1000                                   
                             batch_size: 16                                     
                            num_workers: 2                                      
                            

### Task 2.6 – Sanity check (1,000 iters, with cycle loss)

In [9]:
!python cycle_gan.py \
    --disc patch \
    --use_cycle_consistency_loss \
    --train_iters 1000 \
    --data_preprocess vanilla

                                      Opts                                      
--------------------------------------------------------------------------------
                             image_size: 64                                     
                                   disc: patch                                  
                                    gen: cycle                                  
                             g_conv_dim: 64                                     
                             d_conv_dim: 64                                     
                                   norm: instance                               
             use_cycle_consistency_loss: 1                                      
                              init_type: naive                                  
                            train_iters: 1000                                   
                             batch_size: 16                                     
                            

### Task 2.6 – Full training (10,000 iters, with cycle loss)

In [10]:
!python cycle_gan.py \
    --disc patch \
    --use_cycle_consistency_loss \
    --train_iters 10000 \
    --data_preprocess vanilla

                                      Opts                                      
--------------------------------------------------------------------------------
                             image_size: 64                                     
                                   disc: patch                                  
                                    gen: cycle                                  
                             g_conv_dim: 64                                     
                             d_conv_dim: 64                                     
                                   norm: instance                               
             use_cycle_consistency_loss: 1                                      
                              init_type: naive                                  
                            train_iters: 10000                                  
                             batch_size: 16                                     
                            